In [10]:
from dotenv import find_dotenv
from groq import Groq
from dotenv import load_dotenv
import os
load_dotenv(find_dotenv())
import re

In [11]:

class Provider:
    def __init__(self,name:str,env_var:str,is_free:bool,model:str|None):

        self.name=name
        self.env_var=env_var
        self.is_free=is_free
        self.model=model

PROVIDER = {
    Provider("Groq","GROQ_API_KEY",True,"qwen/qwen3.8-27b")
}

In [16]:
def select_provider()->Provider:
    for p in PROVIDER:
        if os.getenv(p.env_var):
            return p

    expected = "| |".join(p.env_var for p in PROVIDER)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment")

def build_client(provider:Provider) -> Groq:
    api_key = os.getenv(provider.env_var)
    return Groq(api_key=api_key)

def llm_reply(prompt:str, temperature:float=1.0)->str:
    provider = select_provider()
    client = build_client(provider)
    res = client.chat.completions.create(
        model = provider.model,
        messages=[{
            "role":"system",
            "content":prompt,
            
        }],
        temperature=temperature
    )
    return res.choices[0].messages.content

def cot_prompt(prompt:str)->str:
    Prompt = prompt + "think step by step"
    return Prompt


def extract_answer(text:str)->str:

    match = re.findall(r"(?:the answer is | answer is | ####)\s*[:]?\s*([\$A-Za-z0-9\.\,\-]+)",text,re.IGNORECASE)
    if match:
        return match[-1].strip(".$")

    num = re.findall(r"-?\d+(?:\.\d+)?")
    return num[-1] if num else text.strip()


In [14]:
fake_model_answers = [
    # Path 1 (Correct)
    "The juggler has 16 balls. Half of them are golf balls, so 16 / 2 = 8 golf balls. "
    "Half of the golf balls are blue, so 8 / 2 = 4. The answer is 4.",

    # Path 2 (Wrong calculation)
    "There are 16 balls total. Half of them are golf balls, which gives 8 golf balls. "
    "All of these are blue. The answer is 8.",

    # Path 3 (Correct - alternative reasoning)
    "A quarter of all balls are blue golf balls because (1/2) * (1/2) = 1/4. "
    "16 * (1/4) = 4 blue golf balls. The answer is 4.",

    # Path 4 (Correct)
    "16 balls divided by 2 gives 8. Then 8 divided by 2 gives 4. Therefore, the answer is 4.",

    # Path 5 (Wrong)
    "The juggler starts with 16 balls and drops half, leaving 8. The answer is 8."
]

In [18]:
from collections import Counter
from dataclasses import dataclass

@dataclass
class VoteResult:
    winner:str
    tally:dict[str,int]
    samples:int
    confidence:float

def majority_vote(ans:list[str])-> tuple[str,dict[str,int]]:
    if not ans:
        raise ValueError("Ans cannot be empty")

    tally = dict(Counter(ans))
    winner = max(tally,key=tally.get)
    return winner, tally

def self_consistency_answer(sample:list[str])->VoteResult:
    if not sample:
        raise ValueError("Reasoning samples cannot be empty.")


    extracted_ans = [extract_answer(ans) for ans in sample]

    winner, tally = majority_vote(extracted_ans)
    confidence = tally[winner]/len(extracted_ans)

    return VoteResult(
        winner=winner,
        tally=tally,
        samples=len(sample),
        confidence=confidence
    )



In [20]:
res = self_consistency_answer(fake_model_answers)

print("winner",res.winner)
print("Vote tally",res.tally)
print("Number of samples:", res.samples)
print(f"Confidence: {res.confidence:.0%}")

winner 4
Vote tally {'4': 3, '8': 2}
Number of samples: 5
Confidence: 60%
